# Capítulo 6 — RPROP e variantes

Códigos do livro *Backpropagation e Redes Neurais* (Araken de Medeiros Santos).

> **Aviso:** este notebook foi gerado a partir dos scripts do livro e **não foi executado como notebook**; as saídas reportadas no texto vêm da execução dos scripts `.py` correspondentes. Execute as células na ordem, a partir desta pasta, com o ambiente de `requirements.txt` (ver Apêndice E). Valores podem diferir ligeiramente em outras plataformas.

In [ ]:
import os, sys
RAIZ = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(RAIZ, 'codigo'))
os.environ.setdefault('FIGS', os.path.join(RAIZ, 'figs'))
print('rnbook em', os.path.join(RAIZ, 'codigo'))

## Script `cap06/cap06.py`

In [ ]:
"""Cap. 6 - RPROP e variantes: experimentos e figuras (semente fixa; números citados no texto)."""
import sys, os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
from rnbook import MLP
from rnbook import otimizadores as O
from rnbook.estilo import plt, COR, salvar
from rnbook.desenho import caixa, seta

In [ ]:
# ------------------------------------------------------------ 6.1 regra de Manhattan x RPROP em uma calha
H = np.diag([1.0, 10.0]); fq = lambda v: 0.5 * v @ H @ v; gq = lambda v: H @ v
def manhattan(v0, passo, n):
    v = np.array(v0, float); tr = [v.copy()]
    for _ in range(n): v = v - passo * np.sign(gq(v)); tr.append(v.copy())
    return np.array(tr)
tr_m = manhattan([2.5, 1.2], 0.1, 80)
res_r = O.rprop(np.array([2.5, 1.2]), lambda v: fq(v), lambda v: gq(v), variante="irprop-", delta0=0.1, max_iter=80, tol_grad=0)
# reconstruir trajetória do RPROP
tr_r = [np.array([2.5, 1.2])]
v = tr_r[0].copy(); delta = np.full(2, 0.1); g_ant = np.zeros(2)
for _ in range(80):
    g = gq(v); prod = g_ant * g
    delta = np.where(prod > 0, np.minimum(delta * 1.2, 50), delta); delta = np.where(prod < 0, np.maximum(delta * 0.5, 1e-6), delta); g = np.where(prod < 0, 0.0, g)
    v = v - np.sign(g) * delta; g_ant = g; tr_r.append(v.copy())
tr_r = np.array(tr_r)
print("Manhattan (passo 0.1 fixo): distância à origem após 80 passos = %.4f; f = %.4f | menor distância ao longo da trajetória = %.4f" % (np.linalg.norm(tr_m[-1]), fq(tr_m[-1]), min(np.linalg.norm(t) for t in tr_m)))
print("iRPROP- (delta0=0.1, 80 passos): distância final = %.2e; f = %.2e" % (np.linalg.norm(tr_r[-1]), fq(tr_r[-1])))
gx, gy = np.meshgrid(np.linspace(-3, 3, 200), np.linspace(-1.6, 1.6, 200))
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.9)); ax[0].contour(gx, gy, 0.5 * (gx ** 2 + 10 * gy ** 2), levels=[0.5, 2, 5, 10, 16, 25], colors="#CBD5E1", linewidths=0.8)
ax[0].plot(tr_m[:, 0], tr_m[:, 1], "o-", ms=2.5, lw=1, color=COR["amb"], label="Manhattan (passo fixo 0,1)"); ax[0].plot(tr_r[:, 0], tr_r[:, 1], "o-", ms=2.5, lw=1, color=COR["ind"], label="iRPROP− (passos adaptativos)")
ax[0].set_xlim(-3, 3); ax[0].set_ylim(-1.6, 1.6); ax[0].legend(fontsize=8); ax[0].set_title("Só o sinal do gradiente: passo fixo × adaptativo"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
ax[1].semilogy([np.linalg.norm(t) + 1e-16 for t in tr_m], color=COR["amb"], label="Manhattan"); ax[1].semilogy([np.linalg.norm(t) + 1e-16 for t in tr_r], color=COR["ind"], label="iRPROP−"); ax[1].set_xlabel("iteração"); ax[1].set_ylabel("distância à origem"); ax[1].legend(); ax[1].set_title("O passo fixo oscila ao redor do mínimo")
salvar(fig, "f6_manhattan")

In [ ]:
# ------------------------------------------------------------ figura das regras (4 variantes)
fig, ax = plt.subplots(figsize=(11, 4.2)); ax.set_xlim(0, 11); ax.set_ylim(0, 4.2); ax.axis("off"); ax.grid(False)
cols = [("$g_{\\mathrm{ant}}\\,g>0$\n(mesmo sinal)", COR["grn"]), ("$g_{\\mathrm{ant}}\\,g<0$\n(mudou de sinal)", COR["red"]), ("$g_{\\mathrm{ant}}\\,g=0$", COR["gry"])]
for j, (t_, c) in enumerate(cols): caixa(ax, 1.9 + j * 3.0, 3.35, 2.8, 0.75, t_, cor=c, fc="white", fs=9, bold=True)
linhas = [("RPROP+", ["$\\Delta\\leftarrow\\min(\\eta^+\\Delta,\\Delta_{\\max})$\n$\\Delta w=-\\mathrm{sinal}(g)\\Delta$", "$\\Delta\\leftarrow\\max(\\eta^-\\Delta,\\Delta_{\\min})$\nretrocede: $\\Delta w=-\\Delta w_{\\mathrm{ant}}$; $g\\leftarrow0$", "$\\Delta w=-\\mathrm{sinal}(g)\\Delta$"]),
          ("RPROP−", ["$\\Delta\\leftarrow\\min(\\eta^+\\Delta,\\Delta_{\\max})$\n$\\Delta w=-\\mathrm{sinal}(g)\\Delta$", "$\\Delta\\leftarrow\\max(\\eta^-\\Delta,\\Delta_{\\min})$\nsem retrocesso: $\\Delta w=-\\mathrm{sinal}(g)\\Delta$", "$\\Delta w=-\\mathrm{sinal}(g)\\Delta$"]),
          ("iRPROP+", ["$\\Delta\\leftarrow\\min(\\eta^+\\Delta,\\Delta_{\\max})$\n$\\Delta w=-\\mathrm{sinal}(g)\\Delta$", "$\\Delta\\leftarrow\\max(\\eta^-\\Delta,\\Delta_{\\min})$; $g\\leftarrow0$\nse $E_t>E_{t-1}$: retrocede; senão $\\Delta w=0$", "$\\Delta w=-\\mathrm{sinal}(g)\\Delta$"]),
          ("iRPROP−", ["$\\Delta\\leftarrow\\min(\\eta^+\\Delta,\\Delta_{\\max})$\n$\\Delta w=-\\mathrm{sinal}(g)\\Delta$", "$\\Delta\\leftarrow\\max(\\eta^-\\Delta,\\Delta_{\\min})$; $g\\leftarrow0$\n$\\Rightarrow\\Delta w=0$ (sem retrocesso)", "$\\Delta w=-\\mathrm{sinal}(g)\\Delta$"])]
for i, (nm, cel) in enumerate(linhas):
    y0 = 2.55 - i * 0.82; caixa(ax, 0.05, y0, 1.75, 0.7, nm, cor=COR["ind"], fc="#EEF2FF", fs=10, bold=True)
    for j, t_ in enumerate(cel): caixa(ax, 1.9 + j * 3.0, y0, 2.8, 0.7, t_, cor=cols[j][1], fc="white", fs=7.4, lw=1)
salvar(fig, "f6_regras")

In [ ]:
# ------------------------------------------------------------ rede 1-8-1 senoide: variantes (10 sementes)
N = 100; rng = np.random.default_rng(3); X = rng.uniform(0, 1, (N, 1)); T = np.sin(2 * np.pi * X) + rng.normal(0, 0.15, (N, 1))
def roda(var, semente, it=300, **kw):
    r = MLP([1, 8, 1], ["tanh", "identidade"], semente=semente); t0 = time.perf_counter()
    res = O.rprop(r.theta, lambda th: r.perda(th, X, T) / N, lambda th: r.grad(th, X, T) / N, variante=var, max_iter=it, tol_grad=0, **kw); return res, r, time.perf_counter() - t0
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.7)); NOME = {"rprop+": "RPROP+", "rprop-": "RPROP−", "irprop+": "iRPROP+", "irprop-": "iRPROP−"}; cores = {"rprop+": COR["amb"], "rprop-": COR["red"], "irprop+": COR["ind"], "irprop-": COR["grn"]}
fin = {}
for var in O.VARIANTES_RPROP:
    H_ = []; F = []; tt = []
    for s in range(10):
        res, r, dt = roda(var, s); H_.append(res.historico["perda"]); F.append(res.historico["perda"][-1]); tt.append(dt)
    H_ = np.array(H_); fin[var] = F
    ax[0].semilogy(np.median(H_, axis=0), color=cores[var], label=NOME[var], lw=1.8)
    print("%-8s perda final mediana %.4f [mín %.4f, máx %.4f] | época em que a mediana cruza 0.03: %s | tempo %.3f s" % (var, np.median(F), min(F), max(F), (int(np.argmax(np.median(H_, axis=0) < 0.03)) if (np.median(H_, axis=0) < 0.03).any() else None), np.mean(tt)))
ax[0].axhline(0.5 * 0.15 ** 2, color=COR["gry"], ls=":"); ax[0].legend(fontsize=8); ax[0].set_xlabel("época"); ax[0].set_ylabel("perda $E/N$ (mediana de 10 sementes)"); ax[0].set_title("Quatro variantes, rede 1–8–1")
ax[1].boxplot([fin[v] for v in O.VARIANTES_RPROP], tick_labels=[NOME[v] for v in O.VARIANTES_RPROP]); ax[1].set_yscale("log"); ax[1].set_ylabel("perda final $E/N$ (300 épocas)"); ax[1].set_title("Dispersão entre sementes")
salvar(fig, "f6_variantes")

In [ ]:
# ------------------------------------------------------------ passos individuais e mudanças de sinal (iRPROP-)
r = MLP([1, 8, 1], ["tanh", "identidade"], semente=0); th = r.theta.copy(); delta = np.full_like(th, 0.1); g_ant = np.zeros_like(th); D = []; trocas = []; Ltr = []
for k in range(300):
    g = r.grad(th, X, T) / N; prod = g_ant * g; trocas.append(int((prod < 0).sum()))
    delta = np.where(prod > 0, np.minimum(delta * 1.2, 50.0), delta); delta = np.where(prod < 0, np.maximum(delta * 0.5, 1e-6), delta); g = np.where(prod < 0, 0.0, g)
    th = th - np.sign(g) * delta; g_ant = g; D.append(delta.copy()); Ltr.append(r.perda(th, X, T) / N)
D = np.array(D); print("iRPROP-: passos individuais após 300 épocas: mín %.2e mediana %.2e máx %.2e | trocas de sinal por época (média das últimas 100): %.2f de %d pesos" % (D[-1].min(), np.median(D[-1]), D[-1].max(), np.mean(trocas[-100:]), th.size))
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.6))
for j in range(th.size): ax[0].semilogy(D[:, j], lw=0.8, alpha=0.6, color=COR["ind"])
ax[0].set_xlabel("época"); ax[0].set_ylabel("passo individual $\\Delta_i$"); ax[0].set_title("iRPROP−: 25 passos individuais")
ax[1].plot(trocas, color=COR["red"], lw=1); ax[1].set_xlabel("época"); ax[1].set_ylabel("nº de pesos com mudança de sinal"); ax[1].set_title("Mudanças de sinal do gradiente por época")
salvar(fig, "f6_passos")

In [ ]:
# ------------------------------------------------------------ frequência de retrocessos (RPROP+ x iRPROP+)
def conta_retro(var, semente=0, it=300):
    r = MLP([1, 8, 1], ["tanh", "identidade"], semente=semente); th = r.theta.copy(); delta = np.full_like(th, 0.1); g_ant = np.zeros_like(th); dw_ant = np.zeros_like(th); E_ant = np.inf; ret = 0; sc = 0
    for k in range(it):
        g = r.grad(th, X, T) / N; E = r.perda(th, X, T) / N; prod = g_ant * g; pos, neg = prod > 0, prod < 0
        delta = np.where(pos, np.minimum(delta * 1.2, 50.0), delta); delta = np.where(neg, np.maximum(delta * 0.5, 1e-6), delta)
        sc += int(neg.sum())
        if var == "rprop+": dw = np.where(neg, -dw_ant, -np.sign(g) * delta); ret += int(neg.sum())
        else:
            if E > E_ant: dw = np.where(neg, -dw_ant, -np.sign(g) * delta); ret += int(neg.sum())
            else: dw = np.where(neg, 0.0, -np.sign(g) * delta)
        g = np.where(neg, 0.0, g); th = th + dw; g_ant, dw_ant, E_ant = g, dw, E
    return ret, sc
for v in ("rprop+", "irprop+"): print("%s: mudanças de sinal %d | retrocessos efetivos %d (semente 0, 300 épocas, 25 pesos)" % (v, conta_retro(v)[1], conta_retro(v)[0]))

In [ ]:
# ------------------------------------------------------------ 6.4 parada antecipada (rede grande, poucos dados)
def exp_pa(seed, ntr=30, nva=30, H=100, noise=0.4, its=3000, pac=None):
    rng = np.random.default_rng(100 + seed); n = ntr + nva + 300; xs = rng.uniform(0, 1, (n, 1)); ts = np.sin(2 * np.pi * xs) + rng.normal(0, noise, (n, 1))
    xtr, ttr = xs[:ntr], ts[:ntr]; xva, tva = xs[ntr:ntr + nva], ts[ntr:ntr + nva]; xte, tte = xs[ntr + nva:], ts[ntr + nva:]
    r = MLP([1, H, 1], ["tanh", "identidade"], semente=seed, init="he")
    f_tr = lambda th: r.perda(th, xtr, ttr) / len(xtr); f_va = lambda th: r.perda(th, xva, tva) / len(xva)
    res = O.rprop(r.theta, f_tr, lambda th: r.grad(th, xtr, ttr) / len(xtr), variante="irprop-", max_iter=its, tol_grad=0, val=f_va, paciencia=pac)
    rt = lambda th: np.sqrt(np.mean((r.prever(xte, th) - tte) ** 2))
    return r, res, (xtr, ttr, xva, tva), rt
out = []
for seed in range(10):
    r, res, _, rt = exp_pa(seed); hva = np.array(res.historico["perda_val"]); k = int(np.argmin(hva))
    r2, res2, _, rt2 = exp_pa(seed, pac=100)
    out.append((k + 1, rt(res.theta_melhor), rt(res.theta), res2.n_iter, rt2(res2.theta_melhor)))
o = np.array(out)
print("Parada antecipada (rede 1-100-1, 30 treino, 30 validação, 300 teste, ruído 0.4, iRPROP-, 3000 épocas, 10 sementes):")
print("  melhor época (mediana) %d [mín %d, máx %d]" % (np.median(o[:, 0]), o[:, 0].min(), o[:, 0].max()))
print("  RMSE de teste: melhor época %.3f | após 3000 épocas %.3f | melhor época vence em %d de 10 | ruído verdadeiro 0.40" % (o[:, 1].mean(), o[:, 2].mean(), (o[:, 1] < o[:, 2]).sum()))
print("  com paciência 100: parou (mediana) na época %d [mín %d, máx %d]; RMSE de teste médio %.3f" % (np.median(o[:, 3]), o[:, 3].min(), o[:, 3].max(), o[:, 4].mean()))
r, res, (xtr, ttr, xva, tva), rt = exp_pa(0); htr, hva = np.array(res.historico["perda"]), np.array(res.historico["perda_val"]); kbest = int(np.argmin(hva))
print("  semente 0: melhor época %d (val %.4f, treino %.4f); após 3000 épocas: treino %.4f, validação %.4f; RMSE teste %.3f vs %.3f" % (kbest + 1, hva[kbest], htr[kbest], htr[-1], hva[-1], rt(res.theta_melhor), rt(res.theta)))
fig, ax = plt.subplots(1, 2, figsize=(10.4, 3.7)); ax[0].semilogy(htr, color=COR["ind"], label="treino"); ax[0].semilogy(hva, color=COR["amb"], label="validação"); ax[0].axvline(kbest, color=COR["red"], ls="--")
ax[0].text(kbest + 60, hva.max() * 0.6, f"melhor época = {kbest+1}", color=COR["red"], fontsize=9); ax[0].set_xlabel("época"); ax[0].set_ylabel("perda $E/N$"); ax[0].legend(); ax[0].set_title("Parada antecipada: a validação sobe depois do mínimo")
xx = np.linspace(0, 1, 300)[:, None]; ax[1].scatter(xtr, ttr, s=18, color=COR["ind"], label="treino"); ax[1].scatter(xva, tva, s=18, color=COR["amb"], label="validação"); ax[1].plot(xx, np.sin(2 * np.pi * xx), color=COR["gry"], ls=":", label="verdadeira")
ax[1].plot(xx, r.prever(xx, res.theta_melhor), color=COR["grn"], lw=2, label=f"melhor época ({kbest+1})"); ax[1].plot(xx, r.prever(xx, res.theta), color=COR["red"], lw=1.2, label="após 3000 épocas"); ax[1].set_ylim(-2.5, 2.5); ax[1].legend(fontsize=7.5, ncol=2); ax[1].set_xlabel("x"); ax[1].set_title("Rede 1–100–1, 30 pontos de treino")
salvar(fig, "f6_parada")

In [ ]:
# ------------------------------------------------------------ comparação adicional: Friedman (R2 de teste) - 5 sementes
from sklearn.datasets import make_friedman1
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
Xf, yf = make_friedman1(n_samples=400, n_features=10, noise=1.0, random_state=0); Xa, Xte, ya, yte = train_test_split(Xf, yf, test_size=0.25, random_state=0)
esx, esy = StandardScaler().fit(Xa), StandardScaler().fit(ya[:, None]); Xs, Xt, ys = esx.transform(Xa), esx.transform(Xte), esy.transform(ya[:, None]); Nf = len(ys)
for var in O.VARIANTES_RPROP:
    r2 = []; pf = []
    for s in range(5):
        r = MLP([10, 8, 1], ["tanh", "identidade"], semente=s); res = O.rprop(r.theta, lambda th: r.perda(th, Xs, ys) / Nf, lambda th: r.grad(th, Xs, ys) / Nf, variante=var, max_iter=300, tol_grad=0)
        r2.append(r2_score(yte, esy.inverse_transform(r.prever(Xt, res.theta)).ravel())); pf.append(res.historico["perda"][-1])
    print("Friedman #1 %-8s R2 teste mediana %.3f [%.3f, %.3f] | perda treino mediana %.4f" % (var, np.median(r2), min(r2), max(r2), np.median(pf)))
print("OK cap06")

In [ ]:
# ------------------------------------------------------------ figura: organização do código
fig, ax = plt.subplots(figsize=(10.4, 3.9)); ax.set_xlim(0, 10.4); ax.set_ylim(0, 3.9); ax.axis("off"); ax.grid(False)
caixa(ax, 0.2, 2.5, 2.8, 1.1, "Dados\n$X_{tr},T_{tr}$; $X_{va},T_{va}$; $X_{te},T_{te}$", cor=COR["cya"], fc="white", fs=9, bold=True)
caixa(ax, 3.4, 2.5, 3.2, 1.1, "Modelo `MLP`\n$\\theta$ (vetor), `forward`, `perda`,\n`grad`, `jacobiana`", cor=COR["ind"], fc="white", fs=9, bold=True)
caixa(ax, 7.0, 2.5, 3.2, 1.1, "Otimizador\n`rprop`, `quickprop`, `levenberg_marquardt`,\n`delta_bar_delta`, ...", cor=COR["grn"], fc="white", fs=9, bold=True)
caixa(ax, 0.2, 0.5, 4.6, 1.2, "Estado por parâmetro (mesma forma de $\\theta$)\n`delta`, `g_ant`, `dw_ant`  (arrays NumPy)", cor=COR["amb"], fc="white", fs=9, bold=True)
caixa(ax, 5.3, 0.5, 4.9, 1.2, "`Resultado`: `theta`, `theta_melhor`, `historico`\n(`perda`, `perda_val`, `norma_grad`, `tempo`), `n_iter`, `motivo`", cor=COR["vio"], fc="white", fs=9, bold=True)
seta(ax, (3.0, 3.05), (3.4, 3.05)); seta(ax, (6.6, 3.05), (7.0, 3.05)); seta(ax, (8.6, 2.5), (8.0, 1.72)); seta(ax, (2.5, 2.5), (2.5, 1.72), ls="--"); ax.text(3.4, 1.95, "funções `fun(θ)`, `grad(θ)`, `val(θ)`", fontsize=8, color=COR["gry"])
salvar(fig, "f6_estrutura")

## Script `cap06/cap06_1d.py`

In [ ]:
"""Cap. 6: rastreio do passo de cada variante em uma variável (E(w) = 0.5 (w-1)^2 ... e uma função com cruzamento)."""
import numpy as np
def rastro(var, E, g, w0, delta0=0.5, eta_mais=1.2, eta_menos=0.5, dmin=1e-6, dmax=50.0, n=9):
    w, delta, g_ant, dw_ant, E_ant = w0, delta0, 0.0, 0.0, np.inf; linhas = []
    for t in range(1, n + 1):
        gt, Et = g(w), E(w); prod = g_ant * gt; regra = "="
        if prod > 0: delta = min(delta * eta_mais, dmax); regra = ">0"
        elif prod < 0: delta = max(delta * eta_menos, dmin); regra = "<0"
        if var == "rprop+":
            dw = -dw_ant if prod < 0 else -np.sign(gt) * delta
            if prod < 0: gt = 0.0
        elif var == "rprop-": dw = -np.sign(gt) * delta
        elif var == "irprop-":
            if prod < 0: gt = 0.0
            dw = -np.sign(gt) * delta
        elif var == "irprop+":
            if prod < 0:
                dw = -dw_ant if Et > E_ant else 0.0; gt = 0.0
            else: dw = -np.sign(gt) * delta
        linhas.append((t, w, g(w), E(w), regra, delta, dw)); w = w + dw; g_ant, dw_ant, E_ant = gt, dw, Et
    return linhas, w
E = lambda w: 0.5 * (w - 1.0) ** 2 + 0.0; g = lambda w: w - 1.0
for var in ("rprop-", "rprop+", "irprop-", "irprop+"):
    L, wf = rastro(var, E, g, 3.0, n=10)
    print(var); [print("  t=%2d w=%+.4f g=%+.4f E=%.4f prod %s delta=%.4f dw=%+.4f" % l) for l in L]; print("  w final %.4f" % wf)

## Exercícios práticos e projeto: soluções de referência

### 1

Implemente o iRPROP− **sem usar** o pacote `rnbook`, para minimizar funções de várias variáveis, e teste-o em uma quadrática mal condicionada.

1. Escreva `irprop_menos(grad, x0, delta0=0.1, eta_mais=1.2, eta_menos=0.5, delta_min=1e-6, delta_max=50.0, max_iter=500, tol=1e-6)` com estado por parâmetro (`delta`, `g_ant`) e operações vetorizadas.
2. Aplique a $f(\mathbf{x})=\tfrac12\sum_i\lambda_ix_i^2$ com $\lambda=(1;\,10;\,100;\,1000)$ e $\mathbf{x}_0=(1,1,1,1)$.
3. Compare o número de iterações até $\|\mathbf{x}\|<10^{-6}$ com o do gradiente descendente com a melhor taxa estável $\eta=1/\lambda_{\max}$ (limitado a 20 000 iterações).

**Pergunta:** o que isso mostra sobre o condicionamento ($\kappa=1000$) e sobre o RPROP?

In [ ]:
import numpy as np

def irprop_menos(grad, x0, delta0=0.1, eta_mais=1.2, eta_menos=0.5, delta_min=1e-6, delta_max=50.0,
                 max_iter=500, tol=1e-6):
    x = np.array(x0, float)
    delta = np.full_like(x, delta0)
    g_ant = np.zeros_like(x)
    for k in range(1, max_iter + 1):
        if np.linalg.norm(x) < tol:
            return x, k - 1
        g = grad(x)
        prod = g_ant * g
        delta = np.where(prod > 0, np.minimum(delta * eta_mais, delta_max), delta)
        delta = np.where(prod < 0, np.maximum(delta * eta_menos, delta_min), delta)
        g = np.where(prod < 0, 0.0, g)               # iRPROP-: zera o gradiente na mudança de sinal
        x = x - np.sign(g) * delta
        g_ant = g
    return x, max_iter

lam = np.array([1.0, 10.0, 100.0, 1000.0])
grad = lambda x: lam * x
x, k = irprop_menos(grad, np.ones(4))
print("iRPROP-: %d iterações, norma final %.2e" % (k, np.linalg.norm(x)))

x = np.ones(4); eta = 1.0 / lam.max(); k_gd = 0
while np.linalg.norm(x) >= 1e-6 and k_gd < 20000:
    x = x - eta * grad(x); k_gd += 1
print("gradiente descendente (eta=1/1000): %d iterações, norma final %.2e" % (k_gd, np.linalg.norm(x)))

### 2

Estude o efeito de `early_stopping` em `MLPRegressor` em uma base pequena e ruidosa.

1. Gere 150 amostras com `make_friedman1(noise=2.0, random_state=0)`, separe 40% para teste e padronize $X$ e $y$ com o treino.
2. Treine 5 sementes de `MLPRegressor(hidden_layer_sizes=(100,), activation="tanh", solver="adam", alpha=0, learning_rate_init=0.005, max_iter=2000)` **sem** e **com** `early_stopping=True` (`validation_fraction=0.2`, `n_iter_no_change=30`).
3. Relate o $R^2$ de teste (mediana e intervalo) e o número de iterações (`n_iter_`).

**Pergunta:** por que o parâmetro `early_stopping` só existe para `sgd` e `adam` e não para `lbfgs`? Quais cuidados são necessários com a validação interna?

In [ ]:
import warnings
import numpy as np
from sklearn.datasets import make_friedman1
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
X, y = make_friedman1(n_samples=150, n_features=10, noise=2.0, random_state=0)
X_a, X_te, y_a, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
esx, esy = StandardScaler().fit(X_a), StandardScaler().fit(y_a[:, None])
Xs, Xt, ys = esx.transform(X_a), esx.transform(X_te), esy.transform(y_a[:, None]).ravel()

for es in (False, True):
    r2, its = [], []
    for s in range(5):
        m = MLPRegressor(hidden_layer_sizes=(100,), activation="tanh", solver="adam", alpha=0.0,
                         learning_rate_init=0.005, max_iter=2000, early_stopping=es,
                         validation_fraction=0.2, n_iter_no_change=30, random_state=s).fit(Xs, ys)
        r2.append(r2_score(y_te, esy.inverse_transform(m.predict(Xt)[:, None]).ravel())); its.append(m.n_iter_)
    print("early_stopping=%-5s R2 de teste: mediana %.3f [%.3f; %.3f] | iterações: %s" % (es, np.median(r2), min(r2), max(r2), its))

### 3

Compare as quatro variantes do RPROP (pacote `rnbook`) na base **Breast Cancer Wisconsin (Diagnostic)** (UCI; embutida no scikit-learn) e o `lbfgs` do scikit-learn.

1. Separe 70/30 estratificado (`random_state=0`), padronize com o treino e use alvo 0/1.
2. Rede 30–10–1 (tanh, saída sigmoide), erro quadrático médio, 300 épocas, 5 sementes, para cada variante.
3. Registre a acurácia de teste (mediana e intervalo), a perda de treino e o tempo. Compare com `MLPClassifier(hidden_layer_sizes=(10,), activation="tanh", solver="lbfgs", max_iter=300)`.

**Pergunta:** as diferenças entre as variantes são significativas?

In [ ]:
import time
import warnings
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from rnbook import MLP, otimizadores as O

warnings.filterwarnings("ignore")
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
esc = StandardScaler().fit(X_tr)
Xs, Xt = esc.transform(X_tr), esc.transform(X_te)
T = y_tr[:, None].astype(float); N = len(y_tr)

for var in O.VARIANTES_RPROP:
    acc, perda, tempo = [], [], []
    for s in range(5):
        r = MLP([30, 10, 1], ["tanh", "sigmoide"], semente=s)
        t0 = time.perf_counter()
        res = O.rprop(r.theta, lambda th: r.perda(th, Xs, T) / N, lambda th: r.grad(th, Xs, T) / N,
                      variante=var, max_iter=300, tol_grad=0)
        tempo.append(time.perf_counter() - t0)
        acc.append(np.mean((r.prever(Xt, res.theta).ravel() > 0.5) == y_te)); perda.append(res.historico["perda"][-1])
    print("%-8s acurácia teste: mediana %.4f [%.4f; %.4f] | perda de treino %.5f | tempo %.3f s"
          % (var, np.median(acc), min(acc), max(acc), np.median(perda), np.mean(tempo)))
sk = [MLPClassifier(hidden_layer_sizes=(10,), activation="tanh", solver="lbfgs", max_iter=300, random_state=s).fit(Xs, y_tr).score(Xt, y_te) for s in range(5)]
print("sklearn lbfgs acurácia teste: mediana %.4f [%.4f; %.4f]" % (np.median(sk), min(sk), max(sk)))

### RPROP com parada antecipada em dígitos manuscritos

**Objetivo.** Treinar com iRPROP− uma rede para a base Digits, comparando treino com número fixo de épocas e parada antecipada, e comparar com o scikit-learn.

**Dados.** **Digits** do scikit-learn (1797 imagens $8\times8$; subconjunto da base *Optical Recognition of Handwritten Digits* da UCI), 10 classes.

**Tarefas.**
1. Separe 60% treino, 20% validação e 20% teste (estratificado, `random_state=0`); padronize com o treino; alvos *one-hot*.
2. Rede 64–30–10 (tanh e sigmoide), erro quadrático médio, iRPROP−, com (a) 500 épocas fixas e (b) 500 épocas com parada antecipada (paciência 30, validação por perda). 3 sementes.
3. Registre a acurácia de teste nos casos (a) e (b), a época em que a validação foi mínima e o tempo.
4. Compare com `MLPClassifier(hidden_layer_sizes=(30,), activation="tanh", solver="adam", early_stopping=True, max_iter=500)` (3 sementes).
5. Redija 10 a 12 linhas: o papel da validação, o risco de usar o mesmo conjunto para parar e para relatar, e os limites do estudo.

**Para ir além (não testado no livro).** Repita com o conjunto original *optdigits* (UCI) e com o MNIST (Kaggle), observando a necessidade de minilotes (RPROP exige o lote inteiro) e o aumento do custo por época.

In [ ]:
import time
import warnings
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from rnbook import MLP, otimizadores as O

warnings.filterwarnings("ignore")
X, y = load_digits(return_X_y=True)
X_a, X_te, y_a, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_a, y_a, test_size=0.25, random_state=0, stratify=y_a)
esc = StandardScaler().fit(X_tr)
Xs, Xv, Xt = esc.transform(X_tr), esc.transform(X_va), esc.transform(X_te)
T, Tv = np.eye(10)[y_tr], np.eye(10)[y_va]; N, Nv = len(y_tr), len(y_va)

def rprop_digitos(semente, paciencia):
    r = MLP([64, 30, 10], ["tanh", "sigmoide"], semente=semente)
    t0 = time.perf_counter()
    res = O.rprop(r.theta, lambda th: r.perda(th, Xs, T) / N, lambda th: r.grad(th, Xs, T) / N,
                  variante="irprop-", max_iter=500, tol_grad=0,
                  val=lambda th: r.perda(th, Xv, Tv) / Nv, paciencia=paciencia)
    dt = time.perf_counter() - t0
    acc = np.mean(np.argmax(r.prever(Xt, res.theta_melhor), axis=1) == y_te)
    return acc, int(np.argmin(res.historico["perda_val"])) + 1, res.n_iter, dt

print("(a) 500 épocas fixas / (b) parada antecipada com paciência 30")
for rot, pac in (("(a)", None), ("(b)", 30)):
    R = [rprop_digitos(s, pac) for s in range(3)]
    print("%s acurácia teste %s | época de menor validação %s | épocas executadas %s | tempo %.2f s"
          % (rot, np.round([r[0] for r in R], 4), [r[1] for r in R], [r[2] for r in R], np.mean([r[3] for r in R])))
# nota: em (a), 'theta_melhor' é o último theta apenas se não houver val; aqui há val, então o melhor é restaurado nos dois casos

def so_ultimo(semente):
    r = MLP([64, 30, 10], ["tanh", "sigmoide"], semente=semente)
    res = O.rprop(r.theta, lambda th: r.perda(th, Xs, T) / N, lambda th: r.grad(th, Xs, T) / N,
                  variante="irprop-", max_iter=500, tol_grad=0)
    return np.mean(np.argmax(r.prever(Xt, res.theta), axis=1) == y_te)
print("500 épocas, sem validação (últimos parâmetros): acurácia teste", np.round([so_ultimo(s) for s in range(3)], 4))

sk = []
for s in range(3):
    t0 = time.perf_counter()
    m = MLPClassifier(hidden_layer_sizes=(30,), activation="tanh", solver="adam", early_stopping=True,
                      max_iter=500, random_state=s).fit(np.vstack([Xs, Xv]), np.concatenate([y_tr, y_va]))
    sk.append((m.score(Xt, y_te), m.n_iter_, time.perf_counter() - t0))
print("sklearn adam + early_stopping: acurácia teste %s | n_iter_ %s | tempo %.2f s"
      % (np.round([a[0] for a in sk], 4), [a[1] for a in sk], np.mean([a[2] for a in sk])))